In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
%load_ext autoreload
%autoreload 2

In [ ]:
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

In [ ]:
base_dir = "../data"
file_name = "RAG university.pdf"
file_path = f"{base_dir}/raw/{file_name}"

processed_folder_path = f"{base_dir}/processed/{file_name.split('.')[0]}"

elements = partition_pdf(
    filename=file_path,
    strategy="hi_res",
    extract_image_block_types=["Image", "Table"],
    extract_image_block_to_payload=False,
    extract_image_block_output_dir=processed_folder_path,
)
print(len(elements)) 

In [ ]:
from pprint import pprint
for ele in elements: 
    if ele.to_dict()['type'] in ['Image','Table']:
        pprint(ele.to_dict()['metadata'])
        print(ele.to_dict()['text'])
        print('-' * 80)
    

In [ ]:
from pprint import pprint

types = set([ele.to_dict().get('type') for ele in elements])
print(types)

In [ ]:
processed_elements = [ele for ele in elements if ele.to_dict().get('type') not in ['UncategorizedText', 'Header']]

In [ ]:
processed_elements = [ele for ele in processed_elements if len(ele.to_dict().get('text')) > 1]

In [ ]:
types = set([ele.to_dict().get('type') for ele in processed_elements])
print(types)

In [ ]:
from pprint import pprint
for ele in processed_elements: 
    if ele.to_dict()['type'] in ['FigureCaption']:
        pprint(ele.to_dict()['metadata'])
        print(ele.to_dict()['text'])
        print('-' * 80)
    

In [ ]:
pprint(processed_elements[65].to_dict())

In [ ]:
len(processed_elements)

In [ ]:
text_elements = []
multimodal_elements = [] 
for ele in processed_elements: 
    if ele.to_dict().get('type') in ['Image', 'Table']:
        multimodal_elements.append(ele)
    else:
        text_elements.append(ele)
        
chunks = chunk_by_title(
    text_elements,
    max_characters=3000,
    combine_text_under_n_chars=500,
)
print(len(chunks))

In [ ]:
chunks.extend(multimodal_elements)

In [ ]:
types = set([chunk.to_dict().get('type') for chunk in chunks])
print(types)

In [ ]:
from pprint import pprint
for ele in chunks: 
    if ele.to_dict()['type'] in ['CompositeElement']:
        # pprint(ele.to_dict()['metadata'])
        print(ele.to_dict()['text'])
        print('-' * 80)
    

In [ ]:
chunks[14].to_dict()

In [ ]:
chunks.extend(multimodal_elements)

In [ ]:
chunks = [chunk.to_dict() for chunk in chunks]

In [ ]:
from langchain_core.documents import Document
documents = []
for chunk in chunks:
    source = chunk['metadata'].get('file_directory', '') + '/' + chunk['metadata'].get('filename', '')
    documents.append(Document(page_content=chunk.get('text'), metadata={
        'type': chunk.get('type', ''),
        'id': chunk.get('element_id', ''),
        'source': source,
        'image_path': chunk['metadata'].get('image_path', ''),
        'page_number': chunk['metadata'].get('page_number', ''),
        'table_html': chunk['metadata'].get('text_as_html', ''),
        'summary': ""
    }))

print(len(documents))

In [ ]:
for doc in documents:
    if doc.metadata.get('type') == 'Image':
        pprint(doc.metadata['image_path'])
        print("-" * 100)

In [ ]:
documents[1].metadata.keys()

In [ ]:
import pandas as pd
from io import StringIO
def process_html_table(html_content):
    # 1. Parse the HTML into a list of DataFrames
    # pandas uses beautifulsoup4 and lxml under the hood
    tables = pd.read_html(StringIO(html_content))
    
    if not tables:
        return "No tables found."

    # Take the first table
    df = tables[0]

    # 2. Clean the data (Remove empty rows/columns often found in web scrapes)
    df = df.dropna(how='all').reset_index(drop=True)

    # 3. Export to formats optimized for RAG
    # Markdown is best for LLM context
    markdown_version = df.to_markdown(index=False)

    return {
        "markdown": markdown_version,
        "raw_df": df
    }
html_data = [doc.metadata.get("table_html", "") for doc in documents if doc.metadata.get("type", "") == "Table"]
result = process_html_table(html_data[0])
print(result["markdown"])

In [ ]:
result = process_html_table(html_data[1])
print(result["markdown"])

In [ ]:
def summarize_image(chunks):
    """Chunk documents using context-aware text splitting."""
    try:
        llm = ChatGoogleGenerativeAI(
            model="gemini-2.0-flash-lite", 
            temperature=0.1,
            google_api_key=os.getenv("GEMINI_API_KEY"),
        )
        
        system_message = SystemMessage(content="""Analyze this image for retrieval. Provide:
- Main subject/purpose
- Key entities (names, terms, dates, numbers)
- Important relationships or comparisons
- Domain/context
- Concrete facts and data points

Be specific and concise. Focus on searchable information.""")
        
        for chunk in tqdm(chunks, desc="Generating summaries"):
            try:
                content = []
                img_path = chunk.metadata['image_path']
                with open(img_path, 'rb') as f:
                    img_base64 = base64.b64encode(f.read()).decode('utf-8')
                    content.append({
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{img_base64}"}
                    })
            except Exception as e:
                logger.warning(f"Warning: Could not load image {img_path}: {e}")
            
            # Create messages and invoke LLM
            human_message = HumanMessage(content=content)
            response = llm.invoke([system_message, human_message])
            chunk.metadata['summary'] = response.content.strip()
        
        logger.info(f"Successfully generated summaries for {len(chunks)} chunks.")
        return chunks
    except Exception as e:
        logger.error(f"Error generating summaries: {e}")
        raise e

In [ ]:
import logging
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage
from tqdm import tqdm
import os
import base64
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

image_docs = [doc for doc in documents if doc.metadata['type'] == "Image"]

chunks = summarize_image(image_docs)


In [ ]:
for chunk in chunks:
    print(chunk.metadata['summary'])
    print('-' * 80)

# PyMuPDF + pdfplumber

In [ ]:
!pip uninstall pymupdf pdfplumber

In [ ]:
!pip install marker-pdf

In [ ]:
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered


In [ ]:

# Load models (loads once, reuse this object)
converter = PdfConverter(
    artifact_dict=create_model_dict(),
)

# Convert the file
render = converter(file_path)

# Access results
text_content = render.markdown  # Contains text + tables formatted as markdown
images = render.images          # Dictionary of images extracted
metadata = render.metadata

# Save images to disk
import os
output_dir = "processed_images"
os.makedirs(output_dir, exist_ok=True)

for filename, image in images.items():
    image.save(os.path.join(output_dir, filename))

print(text_content)

In [ ]:
import fitz
from PIL import Image
import pytesseract
import io

def fast_pdf_extract(file_path, output_dir):
    doc = fitz.open(file_path)
    results = []
    
    for page_num, page in enumerate(doc):
        # Extract text (very fast)
        text = page.get_text()
        
        # Extract and save images
        for img_index, img in enumerate(page.get_images(full=True)):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            
            # Save image
            image_path = f"{output_dir}/page{page_num}_img{img_index}.png"
            with open(image_path, "wb") as f:
                f.write(image_bytes)
            
            # OCR on image if needed (for text within images)
            image = Image.open(io.BytesIO(image_bytes))
            img_text = pytesseract.image_to_string(image)
            
            results.append({
                'page': page_num,
                'type': 'image',
                'path': image_path,
                'text': img_text
            })
        
        results.append({
            'page': page_num,
            'type': 'text',
            'content': text
        })
    
    return results